# UZHAVAN M4 v1: EfficientNetV2-S training on Colab GPU (notebook v1.1, epoch-level resume)

> **Runtime disconnects resume from `checkpoint_last.pt`; completed seeds are skipped.**
> After a disconnect: reconnect, choose the GPU runtime again, and run all cells. Each seed resumes from its last completed epoch. You never need to type an epoch number.

**Trains:** the approved M4 disease model (6 crops, split `m4_v1_split_seed20260925`) for seeds 1, 2 and 3. The training protocol is identical to v1.0; only checkpointing/resume changed (`train_m4.py` v1.1).
**Does NOT:** evaluate on test or cross-source data. That runs independently on the Mac (`eval_m4.py`).

**Files needed in `MyDrive/uzhavan_m4/`:**
- `m4_v1_colab.zip`: the data package (already uploaded; unchanged)
- `m4_v1_code_v1.1.zip`: the new training code (small). Its `train_m4.py` must have sha256 `96bbd87b270882f7…`

**Per-seed layout** (stable, one folder per seed; a resumed seed continues in the same folder):
`MyDrive/uzhavan_m4/runs/UZ-M4-v1.0-<confighash>-s<seed>/` containing `checkpoint_last.pt` (resume state, saved atomically after every epoch), `checkpoint_best.pt`, `history.json`, `experiment.json` (status: running / interrupted / failed / completed) and `classes.json`.
A **Seed 1 run made with the old v1.0 notebook** (a folder with a date in its name) is recognised as completed if it finished, and is **never modified**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/uzhavan_m4'
RUNS = f'{DRIVE}/runs'
os.makedirs(RUNS, exist_ok=True)
print(sorted(os.listdir(DRIVE)))

In [ ]:
# GPU check: stop if there is no CUDA
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

In [ ]:
# Data package -> local disk (fast), verify integrity; then install train_m4.py v1.1 and verify its checksum
import hashlib, json, shutil, zipfile, time
t = time.time()
if not os.path.exists('/content/pkg/manifest.csv'):
    shutil.copy(f'{DRIVE}/m4_v1_colab.zip', '/content/m4_v1_colab.zip')
    zipfile.ZipFile('/content/m4_v1_colab.zip').extractall('/content/pkg')
meta = json.load(open('/content/pkg/package.json'))
assert hashlib.sha256(open('/content/pkg/manifest.csv','rb').read()).hexdigest() == meta['manifest_sha256'], 'manifest checksum mismatch: re-upload m4_v1_colab.zip'
n = len(os.listdir('/content/pkg/images')); assert n == meta['images'], f'{n} images, expected {meta["images"]}'
zipfile.ZipFile(f'{DRIVE}/m4_v1_code_v1.1.zip').extract('train_m4.py', '/content/code')
TRAIN = '/content/code/train_m4.py'
tsha = hashlib.sha256(open(TRAIN,'rb').read()).hexdigest()
assert tsha == '96bbd87b270882f793ebc432bcfce9adb396befbcc898a51133008a599b6abff', 'train_m4.py checksum mismatch: re-upload m4_v1_code_v1.1.zip'
print('package OK:', n, 'images | train_m4.py v1.1 OK', tsha[:16], f'| {time.time()-t:.0f}s')

In [ ]:
# 1) GPU smoke test (tiny subset, 2 batches). Writes ONLY to local /content/smoke, never to Drive runs
!rm -rf /content/smoke && python {TRAIN} --pkg /content/pkg --out /content/smoke --seed 1 --epochs 1 --bs 32 --workers 2 --smoke

In [ ]:
# 2) Status check: what will happen for each seed (read-only)
import glob, json, torch
def seed_dirs(seed):
    return sorted(d for d in glob.glob(f'{RUNS}/UZ-M4-v1.0-*-s{seed}') if os.path.isdir(d))
def seed_state(seed):
    info = []
    for d in seed_dirs(seed):
        ej = f'{d}/experiment.json'; st = json.load(open(ej)) if os.path.exists(ej) else {}
        status = st.get('status', 'no experiment.json')
        legacy = status == 'experiment'                           # completed with the v1.0 (pre-resume) code
        ck = f'{d}/checkpoint_last.pt'
        last = torch.load(ck, map_location='cpu', weights_only=False)['last_completed_epoch'] if os.path.exists(ck) else None
        info.append(dict(dir=d, status=('completed (v1.0 legacy)' if legacy else status), checkpoint_last=os.path.exists(ck),
                         last_completed_epoch=last, best_val_macro_f1=st.get('best_val_mean_macro_f1'),
                         completed=legacy or status == 'completed'))
    return info
for seed in (1, 2, 3):
    info = seed_state(seed)
    if not info: print(f'seed {seed}: no run yet -> will START at epoch 0'); continue
    for i in info:
        nxt = 'SKIP (completed)' if i['completed'] else (f"RESUME at epoch {i['last_completed_epoch'] + 1}" if i['checkpoint_last'] else 'no checkpoint -> epoch 0 (in the stable v1.1 folder)')
        print(f"seed {seed}: {os.path.basename(i['dir'])}\n   status={i['status']} | checkpoint_last exists={i['checkpoint_last']} | "
              f"last completed epoch={i['last_completed_epoch']} | best val macro-F1={i['best_val_macro_f1']} | next: {nxt}")

In [ ]:
# 3) Three-seed experiment. Re-run after ANY disconnect: completed seeds are skipped, unfinished seeds resume from checkpoint_last.pt
import datetime, signal
for seed in (1, 2, 3):
    if any(i['completed'] for i in seed_state(seed)):
        print(f'seed {seed}: completed -> skipped'); continue
    print(f'=== seed {seed} (resume auto) ===', flush=True)
    log = f'{RUNS}/train_seed{seed}_{datetime.datetime.now():%Y%m%d%H%M}.log'
    cmd = ['python', TRAIN, '--pkg', '/content/pkg', '--out', RUNS, '--seed', str(seed),
           '--epochs', '30', '--bs', '32', '--workers', '2', '--resume', 'auto']
    with open(log, 'w') as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in p.stdout: print(line, end=''); fh.write(line); fh.flush()
            rc = p.wait()
        except KeyboardInterrupt:                                  # manual stop: let the trainer mark 'interrupted' cleanly
            p.send_signal(signal.SIGTERM); p.wait(); print('stopped; rerun this cell to resume'); raise
    assert rc == 0, f'seed {seed} exited with {rc}: see {log}. Rerun this cell to resume from checkpoint_last.pt'

In [ ]:
# 4) Summary: then download MyDrive/uzhavan_m4/runs/ to the Mac for independent evaluation
for seed in (1, 2, 3):
    for i in seed_state(seed):
        print(f"seed {seed}: {os.path.basename(i['dir'])} | {i['status']} | last epoch {i['last_completed_epoch']} | best val macro-F1 {i['best_val_macro_f1']}")